# Agentic AI Applications — Masterclass Part 2 of 3: Protocols & Coding Agents

Second of three notebooks in the agentic AI masterclass. N1 covered foundations (the agent loop, memory, planning, HITL) and multi-agent orchestration. This notebook tackles two adjacent topics: **how agents talk to tools (and to each other)** and **what makes coding agents different.**

| | Notebook | Scope |
|---|---|---|
| N1 | `02a_agents_foundations_orchestration.ipynb` | Single-agent fundamentals → memory → multi-agent → DeepAgents |
| **N2** | **`02b_agents_protocols_coding.ipynb`** *(this one)* | MCP, A2A, coding agents |
| N3 | `02c_agents_eval_serving_ops.ipynb` | Evaluation, serving, monitoring, guardrails, maintenance |

## What this notebook covers

**Part A — Protocols (§1-5)**: why the field needed standards in the first place, the architecture and lifecycle of MCP (Model Context Protocol), the A2A v1.0 protocol for agent-to-agent communication, and a decision matrix for picking among MCP / A2A / plain function calling.

**Part B — Coding agents (§6-10)**: what makes coding agents structurally different from chat agents, the four architectural paradigms that dominate 2026 (autonomous cloud / agent-first IDE / terminal / spec-driven), the canonical tool kit (`view`, `edit`, `bash`, `glob`, `grep`, `str_replace`, `run_tests`), context management for large codebases, and the failure modes you'll be asked about in interviews.

## Why protocols and coding agents are paired

Two reasons. **First**, MCP was originally built for coding agents — Anthropic's first MCP host was Claude Desktop with filesystem/bash/git servers, and the second was Claude Code. Anything you read about MCP today still skews toward developer-tool servers. **Second**, the four coding-agent architectures all use MCP heavily — even the autonomous cloud ones (Devin) and the IDE ones (Cursor, Antigravity). Understanding MCP without understanding what it's used for is hollow. Understanding coding agents without MCP is incomplete.

## Stack assumptions (2026)

- **MCP**: November 2025 spec, Streamable HTTP transport, OAuth 2.1 auth, donated to Linux Foundation's Agentic AI Foundation in December 2025. SDK downloads ~97M/month.
- **A2A**: v1.0 (early 2026) — signed agent cards, gRPC support, multi-tenancy. Linux Foundation governance since June 2025. 150+ organizations in production.
- **Coding agents**: Claude Code (terminal), Cursor + Google Antigravity (agent-first IDEs), Devin (autonomous cloud), Augment / Intent (spec-driven). Antigravity was released Nov 2025 alongside Gemini 3.
- **Python libs**: `mcp` (FastMCP), `a2a-sdk` for protocol implementations; everything else from N1 still applies.

## How to read this notebook

Code-lighter than N1 — protocols are mostly architectural, and we won't run real MCP servers in a notebook (they're separate processes by design). Most cells are markdown with diagrams and decision matrices. The few code cells show **schema-shaped examples**: what an agent card looks like, what a minimal MCP server looks like in Python, what the tool calls actually carry over the wire.

## Setup


In [1]:
# Install the libraries used in this notebook (none strictly required to run — most cells are markdown).
# !pip install -q mcp fastmcp a2a-sdk httpx pydantic
# !pip install -q langchain-mcp-adapters  # if you want to use MCP tools inside LangGraph

# Optional config:
# import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # for tool-call demos
# os.environ["OPENAI_API_KEY"] = "sk-..."

import json
print("Setup cell. The protocol demos below show schemas and config files — no servers run inside the notebook.")


Setup cell. The protocol demos below show schemas and config files — no servers run inside the notebook.


---
# Part A: Protocols

## §1 — The protocols problem: why standards matter

Step back to 2023 for a second. You wanted your LLM to use a tool. Your options were:

1. **Prompt the LLM in ReAct format and parse free-form text.** Brittle, slow, full of regex edge cases. Worked, barely.
2. **Use the host's proprietary function-calling API.** OpenAI had one shape, Anthropic another, Google a third. Tool definitions weren't portable.

By 2024 every team building serious AI tools was solving the same problem twice: once for OpenAI clients, once for Anthropic clients, once for whatever Cursor/Continue/Cline/local-models needed. **Every integration was M × N** — M hosts, N tools.

That's the problem MCP was designed to solve.

### The N+M insight

The trick is to insert a thin protocol layer between hosts and tools:

```
Without MCP (M × N integrations):           With MCP (M + N integrations):
                                            
   Claude  ─┬─ Notion                          Claude  ─┐
   ChatGPT ─┼─ GitHub                          ChatGPT ─┤
   Cursor  ─┼─ Stripe                          Cursor  ─┼─→ [MCP] ←─┬─ Notion server
   Custom  ─┴─ Linear                          Custom  ─┘           ├─ GitHub server
                                                                     ├─ Stripe server
   each host integrates each tool              host speaks MCP;      └─ Linear server
   M × N = 16 integrations                     each server speaks MCP
                                               M + N = 8 integrations
```

USB-C is the standard analogy. Before USB-C, every device had its own port; now any device that speaks USB-C plugs into any host that speaks USB-C. **MCP is USB-C for AI.**

### Why "yet another protocol"?

Skeptics in 2024 asked: why not just use REST? gRPC? GraphQL?

The protocol layer needed two things REST didn't natively provide:

1. **A standard way to describe capabilities** — what tools exist, what they do, what arguments they take. REST APIs have OpenAPI, but every team writes their own OpenAPI and there's no convention for "this is an LLM-callable tool."
2. **Bidirectional, stateful, streaming interaction** — the host needs to *notify* the server (resource changed), the server needs to *stream* partial results (long tool runs), and both sides need to negotiate capabilities at session start.

MCP picked **JSON-RPC 2.0 over pluggable transports** (stdio for local, Streamable HTTP for remote). JSON-RPC is half a page of spec. Streamable HTTP is HTTP + SSE for streams. Together they cover both local desktop integrations and remote enterprise servers without inventing anything new.

### Two protocols, two scopes

By early 2026, two protocols matter:

| Protocol | Scope | Originated |
|---|---|---|
| **MCP** (Model Context Protocol) | Tools and data — how an LLM uses external capabilities | Anthropic, Nov 2024; donated to Linux Foundation Dec 2025 |
| **A2A** (Agent2Agent) | Agent-to-agent — how an agent delegates to another agent across vendor boundaries | Google, Apr 2025; donated to Linux Foundation Jun 2025 |

The official line: **MCP for tools, A2A for agents.** Pick one based on whether the thing you're calling is a passive tool (it does what you ask, returns a result) or an active agent (it has its own loop, its own tools, makes its own decisions, can take time, can stream updates). We'll get to the decision matrix in §5.

### Why this matters for your interviews

Interviewers in 2026 ask about MCP because it's the layer where most AI engineering work happens — building MCP servers for company tools, deploying them at scale, securing them with OAuth, debugging tool-selection issues. **"Have you built an MCP server?"** is the same kind of question 2018 interviewers asked about REST APIs. The answer "yes, and here's why I made it stateless" lands well.

A2A is more recent but gaining ground fast. Worth knowing the shape — agent cards, task lifecycle, signed discovery — even if you haven't built one yet.


## §2 — MCP architecture: host / client / server, and the three primitives

MCP has a tiny vocabulary. Once you have these terms straight, every MCP doc reads cleanly.

### The three roles

| Role | What it is | Examples |
|---|---|---|
| **Host** | The AI application that initiates connections | Claude Desktop, Claude Code, Cursor, VS Code Copilot, ChatGPT Desktop, Antigravity |
| **Client** | A session manager inside the host, **one per server** | Internal component of the host; you don't write these |
| **Server** | A process that exposes capabilities over MCP | `github-mcp`, `notion-mcp`, `filesystem-mcp`, your custom server |

A single host runs **N clients**, each connected to **one server**. Each client handles its own handshake, capability discovery, request/response.

```
   ┌──────────────────────────────────────┐
   │             HOST                      │
   │  (Claude Desktop / Cursor / etc.)     │
   │                                       │
   │  ┌────────┐  ┌────────┐  ┌────────┐   │
   │  │client 1│  │client 2│  │client 3│   │
   │  └────┬───┘  └────┬───┘  └────┬───┘   │
   └───────┼───────────┼───────────┼───────┘
           │           │           │
           ▼           ▼           ▼
       [server 1]  [server 2]  [server 3]
        github      notion      filesystem
```

The user interacts with the host. The host's LLM emits tool calls. The host routes each call to the right client, the client sends it to its server, the server runs it, the result comes back. From the user's perspective, the experience is: "the LLM has tools." Under the hood, those tools live in independent processes the host doesn't have to know anything about.

### The three primitives

Each MCP server can expose any combination of three primitives:

| Primitive | Nature | What the LLM does with it | Examples |
|---|---|---|---|
| **Tools** | Executable actions with side effects | Calls them with arguments to get results | `send_email`, `create_issue`, `run_query`, `git_commit` |
| **Resources** | Read-only data | Reads them as context, doesn't "call" them | Files, DB tables, web pages, system status |
| **Prompts** | Reusable templates / starter flows | The host surfaces them as commands the user can pick | "Summarize this PR for release notes", "Onboard a new contractor" |

A clean mental model:

> **Resources are nouns** (data the LLM can read).  
> **Tools are verbs** (actions the LLM can take).  
> **Prompts are recipes** (parameterized workflows the user invokes).

Most production servers are heavy on tools, moderate on resources, light on prompts.

### What the protocol traffic actually looks like

Every message is JSON-RPC 2.0 — request, response, or notification. A tool call has three messages on the wire:

```jsonc
// 1. Host's client sends a request
{"jsonrpc":"2.0","id":42,"method":"tools/call",
 "params":{"name":"create_issue",
           "arguments":{"title":"Fix login bug","priority":"high"}}}

// 2. Server returns a response
{"jsonrpc":"2.0","id":42,
 "result":{"content":[{"type":"text","text":"Issue #1234 created"}]}}

// 3. (Optional) Server can send progress notifications during long tools
{"jsonrpc":"2.0","method":"notifications/progress",
 "params":{"progressToken":"abc","progress":0.5,"total":1.0}}
```

The `id` ties response to request. Notifications (no `id`) flow either direction without requiring a response. That's the whole wire protocol.

### Capability negotiation: handshake at session start

When a client connects to a server, they exchange capabilities first:

```
Client → Server:  initialize(protocolVersion="2025-11-25",
                              capabilities={...client supports...})
Server → Client:  initialize result(protocolVersion="2025-11-25",
                                     capabilities={tools:{...}, resources:{...}, prompts:{}})
```

Either side can advertise which features it supports — streaming, subscriptions, sampling (the server asking the host's LLM to complete something), and so on. **Negotiated capabilities determine what's legal to call later.** If a server didn't advertise `prompts`, the host won't list any prompts to the user.

### Resources vs. tool returns — a distinction that catches people

A resource is a **persistent, addressable piece of data** the LLM can attach to its context. Like a file URI: `weather://Delhi/current`. The client can subscribe to changes and the server can push updates.

A tool *return value* is **the result of an action**. It's not addressable, it doesn't persist, you don't subscribe to it.

When in doubt: if the same lookup keeps coming back (logs, DB rows, file contents that change) and the LLM might want it on every turn → expose it as a resource. If it's a one-off action with a one-off result (send this email, run this query right now) → tool.


## §3 — Transport, lifecycle, OAuth 2.1, and a minimal MCP server

### The two official transports

MCP has **two transports, deliberately**. The November 2025 spec explicitly resisted adding more — keeping the spec small was a design value.

| Transport | When to use | Mechanics |
|---|---|---|
| **stdio** | Local execution (desktop apps, dev) | Server is a subprocess; messages flow over the server's stdin/stdout as newline-delimited JSON-RPC |
| **Streamable HTTP** | Remote services, production, multi-user | Single HTTP endpoint that handles both POST (requests) and GET (SSE stream); supports stateful sessions and bidirectional notifications |

A few hard-won facts you'll be asked about:

- **HTTP+SSE was deprecated** in March 2025 in favor of Streamable HTTP. If you see a tutorial mentioning a separate SSE endpoint, it's pre-2025 — out of date.
- **Streamable HTTP supports both stateful and stateless modes.** Stateful uses an `Mcp-Session-Id` header; stateless is a 2026 evolution to fix horizontal-scaling issues with stateful sessions stuck on one pod.
- **The 2026 roadmap explicitly says no new transports will be added this cycle** — work focuses on evolving Streamable HTTP (scalability, stateless sessions, `.well-known` metadata).

### The session lifecycle

```
1. Initialize
   client → server:  initialize(protocolVersion, capabilities)
   server → client:  initialize result(capabilities)
   client → server:  initialized (notification, confirming handshake)

2. Discovery (the client asks what's available)
   client → server:  tools/list      → [tool schemas]
   client → server:  resources/list  → [resource URIs]
   client → server:  prompts/list    → [prompt templates]

3. Operation (loop while the session is alive)
   client → server:  tools/call(name, args)              → tool result
   client → server:  resources/read(uri)                 → resource contents
   server → client:  notifications/resources/changed     → resource invalidation
   server → client:  notifications/progress              → progress on long tool

4. Termination
   either side closes the transport; stateful sessions can be resumed via the session ID
```

The "discovery" step is what makes MCP feel cleaner than function calling. The host doesn't hardcode the tool list — it asks the server. New tool added to the server → host sees it on next `tools/list`.

### Authentication: OAuth 2.1 (June 2025 spec)

For remote MCP servers (Streamable HTTP), the auth standard is **OAuth 2.1 with PKCE**. The June 2025 spec added Resource Indicators (RFC 8707) — without these, a rogue server could trick the host into leaking tokens for other services.

The flow:

```
1. User opens host (e.g. Cursor) and adds a remote MCP server URL
2. Host fetches /.well-known/oauth-authorization-server on the MCP server's domain
3. Host opens a browser for OAuth — user signs in (SSO usually)
4. Host receives tokens, stores them securely
5. Host attaches Authorization: Bearer <token> on every MCP request
6. Token refresh happens silently
```

In practice: most enterprise MCP deployments now sit behind their company SSO, and the user goes through the same auth flow they would for any other internal tool.

### A minimal MCP server in Python (FastMCP)

The Python SDK's FastMCP makes a server two decorators:

```python
# Save this as weather_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("weather-server")

@mcp.tool()
def get_weather(city: str, date: str) -> dict:
    """Get weather for a city on a given date.

    Args:
        city: City name, e.g. 'Delhi' or 'Bangalore'
        date: ISO date string YYYY-MM-DD
    """
    # Real implementation would call OpenWeather or similar
    return {"city": city, "date": date, "high": 38, "condition": "clear"}

@mcp.resource("weather://{city}/current")
def current_weather(city: str) -> str:
    """A live weather resource; the host can subscribe to changes."""
    return f"Current weather in {city}: 32°C, partly cloudy"

@mcp.prompt()
def trip_briefing(destination: str) -> str:
    """A reusable prompt template the user can invoke."""
    return f"Brief me on travel conditions for {destination}, including weather, visa, currency."

if __name__ == "__main__":
    mcp.run(transport="stdio")     # for production remote: transport="streamable-http"
```

And on the host side — Claude Desktop's config, for example — you point it at this server:

```json
{
  "mcpServers": {
    "weather": {
      "command": "python",
      "args": ["/path/to/weather_server.py"]
    }
  }
}
```

That's a complete MCP integration. Same server works in Claude Desktop, Cursor, Claude Code, VS Code Copilot — anywhere that speaks MCP.

### Security model (read this carefully)

The spec is blunt: **tool descriptions are untrusted**. A malicious server can put prompt-injection payloads into its tool descriptions, and the host will helpfully feed them to the LLM. Three rules:

1. **Whitelist servers.** Don't auto-trust anything in a public registry. Treat MCP servers like supply-chain software — review before you trust.
2. **Explicit user consent for destructive actions.** The host should pop a confirmation dialog for anything that creates, modifies, or sends.
3. **Sandbox server execution where possible.** Especially for code-execution tools, run them in a container or VM.

> **Interview note.** *"What's the threat model for third-party MCP servers?"* The answer interviewers want: (1) tool descriptions are prompt-injection vectors, (2) servers see everything in the context the host passes them — possible data exfiltration, (3) servers can claim capabilities they don't have safely. Mitigations: whitelist + user consent + sandboxing + audit logs.


In [2]:
# Schemas and JSON-RPC traffic for an MCP tool call.
# We don't spawn a real server here — that would be a separate process. We just show what's on the wire.

import json

# 1. The tool's JSON schema, as the server advertises it via tools/list
tool_schema = {
    "name": "get_weather",
    "description": "Get weather for a city on a given date. Use this for any weather-related query.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'Delhi'"},
            "date": {"type": "string", "format": "date", "description": "YYYY-MM-DD"},
        },
        "required": ["city", "date"],
    },
}
print("=== What the server advertises (tools/list result) ===")
print(json.dumps(tool_schema, indent=2))

# 2. The JSON-RPC request the host's client sends when the LLM emits a tool call
tool_call_request = {
    "jsonrpc": "2.0",
    "id": 42,
    "method": "tools/call",
    "params": {
        "name": "get_weather",
        "arguments": {"city": "Delhi", "date": "2026-05-20"},
    },
}
print("\n=== JSON-RPC request on the wire ===")
print(json.dumps(tool_call_request, indent=2))

# 3. The response the server returns
tool_call_response = {
    "jsonrpc": "2.0",
    "id": 42,
    "result": {
        "content": [
            {"type": "text", "text": json.dumps({"city": "Delhi", "date": "2026-05-20",
                                                   "high": 38, "condition": "clear"})}
        ],
        "isError": False,
    },
}
print("\n=== JSON-RPC response on the wire ===")
print(json.dumps(tool_call_response, indent=2))

# 4. A progress notification (server → client during a long tool call)
progress_notification = {
    "jsonrpc": "2.0",
    "method": "notifications/progress",
    "params": {"progressToken": "weather-fetch-abc", "progress": 0.5, "total": 1.0,
                "message": "Fetching 7-day forecast..."},
}
print("\n=== Progress notification (no `id`, no response expected) ===")
print(json.dumps(progress_notification, indent=2))

print("\nThe whole protocol is JSON-RPC over a transport. That's it.")


=== What the server advertises (tools/list result) ===
{
  "name": "get_weather",
  "description": "Get weather for a city on a given date. Use this for any weather-related query.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "City name, e.g. 'Delhi'"
      },
      "date": {
        "type": "string",
        "format": "date",
        "description": "YYYY-MM-DD"
      }
    },
    "required": [
      "city",
      "date"
    ]
  }
}

=== JSON-RPC request on the wire ===
{
  "jsonrpc": "2.0",
  "id": 42,
  "method": "tools/call",
  "params": {
    "name": "get_weather",
    "arguments": {
      "city": "Delhi",
      "date": "2026-05-20"
    }
  }
}

=== JSON-RPC response on the wire ===
{
  "jsonrpc": "2.0",
  "id": 42,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "{\"city\": \"Delhi\", \"date\": \"2026-05-20\", \"high\": 38, \"condition\": \"clear\"}"
      }
    ],
    

## §4 — A2A protocol v1.0: agent cards, task lifecycle, signed discovery

A2A (Agent2Agent) is the protocol layer for **agents talking to other agents** — not tools. Google introduced it in April 2025; donated it to the Linux Foundation in June 2025; v1.0 shipped in early 2026.

If MCP is "USB-C for AI" (how an LLM uses tools), then A2A is **"HTTP for agents"** (how an agent delegates to another agent across vendor and framework boundaries).

### Why a second protocol? Why not just use MCP?

Because the interaction shape is different. With a tool, the host knows roughly how long it'll take and what shape the result will be. With another agent, the receiver is itself running an agent loop — it might take minutes or hours, stream partial updates, ask clarifying questions, fail and recover. The interaction is **task-oriented, stateful, multi-modal, peer-to-peer**.

The official line from the Google / Linux Foundation A2A docs: **"MCP for tools and data, A2A for agents."** You can do agent-to-agent over MCP — wrap each agent as an MCP tool — but you lose task lifecycle semantics (cancellation, status updates, artifact streaming) that A2A provides natively.

### The agent card — the heart of A2A

Every A2A-compliant agent publishes a JSON document at a well-known URL: `/.well-known/agent-card.json`. This is the discovery primitive.

```jsonc
{
  "name": "research-agent",
  "description": "Research multi-aspect questions and return structured findings.",
  "version": "1.2.0",
  "url": "https://research-agent.example.com/a2a",
  "capabilities": {
    "streaming": true,
    "pushNotifications": true,
    "stateTransitionHistory": true
  },
  "skills": [
    {
      "id": "deep_research",
      "name": "Deep research",
      "description": "Multi-source research with citations on technical topics",
      "inputModes": ["text"],
      "outputModes": ["text", "file/pdf"]
    }
  ],
  "authentication": {"schemes": ["oauth2", "bearer"]},
  "defaultInputModes": ["text"],
  "defaultOutputModes": ["text"]
}
```

This is the "OpenAPI for agents." Another agent fetches it to know what this agent can do, how to authenticate, what input/output modalities it speaks. **Dynamic discovery without code changes** — adding a new remote agent is just adding a new URL.

### Signed agent cards (v1.0)

In v1.0 (early 2026), the spec added **cryptographic signatures on agent cards.** Without this, an attacker could stand up a fake agent card and redirect other agents into a card-forgery attack. With signed cards, a receiving agent can verify the card was actually issued by the domain owner.

This is the trust model that makes decentralized agent discovery viable. Compare: the entire HTTPS web works because cert authorities sign domains; A2A v1.0 brings the same idea to agent discovery.

### Task lifecycle: the protocol's main verb

Where MCP is built around **tool calls** (request/response), A2A is built around **tasks** (long-running stateful units of work). A task moves through states:

```
   submitted ──→ working ──→ ┬─→ completed
                              │
                              ├─→ failed
                              ├─→ canceled
                              │
                              └─→ input-required ──→ working ──→ ...
```

Each state can carry artifacts (intermediate or final results), status updates, and messages back to the client. The client can subscribe to the task's event stream (SSE in v1.0; the v1.0 release added gRPC as an alternative).

A typical task flow:

```
Client (calling agent): tasks/send({"task": "Research LangGraph vs DSPy"})
   → returns task_id, status="submitted"

Server (research agent): starts working, emits events on the task's stream:
   status: "working" + artifact: partial findings
   status: "working" + artifact: more findings
   status: "completed" + artifact: final report (PDF)
```

### Other v1.0 changes

- **Multi-tenancy** — one A2A endpoint can host multiple agents, letting SaaS providers serve different agents per tenant.
- **Multi-protocol bindings** — the same logical agent can be exposed over JSON-RPC and gRPC simultaneously. Picking gRPC matters when you want strict typing and HTTP/2 multiplexing.
- **Version negotiation** — backward-compatible migration from v0.3 to v1.0 is guaranteed at the spec level.

### What an A2A interaction looks like in practice

The Google ADK (Agent Development Kit, 1.0 GA in April 2026) gives you both sides cheaply:

```python
# Exposing your agent as an A2A server (server side)
from google.adk import Agent, expose_as_a2a

class MyResearchAgent(Agent):
    def __init__(self): ...
    async def run(self, task): ...   # your agent's logic

expose_as_a2a(
    MyResearchAgent(),
    agent_card_path="/.well-known/agent-card.json",
    base_url="https://research.mycompany.com/a2a",
)

# Calling a remote A2A agent (client side)
from google.adk import RemoteA2aAgent

remote = RemoteA2aAgent(card_url="https://research.othercompany.com/.well-known/agent-card.json")
task = await remote.send_task("Research the EV market in Southeast Asia")
async for event in task.stream():
    print(event.status, event.artifact)
```

You don't need to use ADK — Python, Java, Go, TypeScript, .NET SDKs all ship; LangGraph has an A2A adapter that wraps a LangGraph agent as an A2A server. The protocol is framework-agnostic.

### When to reach for A2A

| Use case | A2A fit |
|---|---|
| Cross-company agent integration (your agent calls a partner's agent) | Strong fit — that's exactly the design target |
| Multi-vendor multi-agent system (LangGraph + ADK + CrewAI agents talking) | Strong fit — the framework-agnostic interop layer |
| Internal multi-agent inside one codebase | Overkill — use LangGraph supervisor/swarm from N1 |
| Calling a known tool / API | Wrong layer — that's MCP or function calling |
| Long-running task with streaming updates | A2A's task lifecycle is built for this |

> **Interview note.** *"How does A2A relate to MCP?"* They compose. A2A coordinates agents; each of those agents internally uses MCP (and/or function calling) to talk to tools. A 2026 architecture diagram looks like: top layer A2A for cross-agent coordination, middle layer LangGraph / Antigravity / ADK orchestrating internal agents, bottom layer MCP servers wrapping each tool.


In [3]:
# An A2A v1.0 agent card with a signature, plus the task lifecycle messages.
# Same approach as before — we show the JSON on the wire, not a running server.

import json

# An agent card that another agent fetches from /.well-known/agent-card.json
agent_card = {
    "name": "research-agent-pro",
    "description": "Multi-aspect deep research with citations. Hour-long tasks supported.",
    "version": "1.2.0",
    "url": "https://research.example.com/a2a",
    "capabilities": {
        "streaming": True,           # supports SSE event streams
        "pushNotifications": True,   # can call back to client webhooks
        "stateTransitionHistory": True,  # exposes task history
    },
    "skills": [
        {
            "id": "deep_research",
            "name": "Deep research with sources",
            "description": "Multi-source research with structured findings + cited PDF",
            "inputModes": ["text"],
            "outputModes": ["text", "file/pdf"],
            "examples": [
                "Compare EV market growth across SEA countries 2023-2025",
                "Survey of agent-evaluation frameworks for production use",
            ],
        }
    ],
    "authentication": {"schemes": ["oauth2"], "tokenUrl": "https://auth.example.com/token"},
    "defaultInputModes": ["text"],
    "defaultOutputModes": ["text"],
    # v1.0 addition — signed agent card (proves the card was issued by the domain owner)
    "signature": {
        "algorithm": "RS256",
        "signedBy": "research.example.com",
        "signedAt": "2026-05-19T10:00:00Z",
        "signature": "<base64-encoded-RSA-signature-over-canonical-card-bytes>",
    },
}
print("=== Agent card at /.well-known/agent-card.json ===")
print(json.dumps(agent_card, indent=2)[:1200], "...\n")

# A task being sent from one agent to this agent
send_task = {
    "jsonrpc": "2.0",
    "id": "req-1",
    "method": "tasks/send",
    "params": {
        "task": {
            "skill_id": "deep_research",
            "input": {"type": "text",
                       "text": "Compare LangGraph supervisor vs swarm patterns for production"},
        }
    },
}
print("=== Sending a task (client → server) ===")
print(json.dumps(send_task, indent=2), "\n")

# The stream of events the server emits while working
events_stream = [
    {"task_id": "t-abc-123", "status": "submitted", "timestamp": "2026-05-19T10:00:01Z"},
    {"task_id": "t-abc-123", "status": "working", "timestamp": "2026-05-19T10:00:05Z",
     "artifact": {"type": "text", "name": "preliminary_findings",
                   "content": "Surveying LangGraph 1.x docs..."}},
    {"task_id": "t-abc-123", "status": "working", "timestamp": "2026-05-19T10:02:30Z",
     "artifact": {"type": "text", "name": "comparison_table", "content": "..."}},
    {"task_id": "t-abc-123", "status": "completed", "timestamp": "2026-05-19T10:05:00Z",
     "artifact": {"type": "file/pdf", "name": "report.pdf",
                   "uri": "https://research.example.com/artifacts/t-abc-123/report.pdf"}},
]
print("=== Event stream from server (SSE) ===")
for e in events_stream:
    print(f"  [{e['status']:10}] {e['timestamp']}  → "
          f"{e.get('artifact', {}).get('name', '-')}")

print("\nThe protocol carries the WHOLE long-running interaction.")
print("Compare MCP: a single tool call resolves in one request/response.")
print("A2A: a task can run for an hour with continuous updates.")


=== Agent card at /.well-known/agent-card.json ===
{
  "name": "research-agent-pro",
  "description": "Multi-aspect deep research with citations. Hour-long tasks supported.",
  "version": "1.2.0",
  "url": "https://research.example.com/a2a",
  "capabilities": {
    "streaming": true,
    "pushNotifications": true,
    "stateTransitionHistory": true
  },
  "skills": [
    {
      "id": "deep_research",
      "name": "Deep research with sources",
      "description": "Multi-source research with structured findings + cited PDF",
      "inputModes": [
        "text"
      ],
      "outputModes": [
        "text",
        "file/pdf"
      ],
      "examples": [
        "Compare EV market growth across SEA countries 2023-2025",
        "Survey of agent-evaluation frameworks for production use"
      ]
    }
  ],
  "authentication": {
    "schemes": [
      "oauth2"
    ],
    "tokenUrl": "https://auth.example.com/token"
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [


## §5 — MCP vs A2A vs function calling: decision matrix and composition

### The three options at a glance

| | Function calling | MCP | A2A |
|---|---|---|---|
| **What you're talking to** | A function inside your app | A tool exposed by an external (or co-located) MCP server | Another autonomous agent, often owned by someone else |
| **State** | Stateless request/response | Session-stateful; can stream and notify | Task-stateful; long-running with lifecycle states |
| **Where the tool lives** | Same process as your code | Separate process (often a separate service / machine) | Separate agent, often a separate org |
| **Discovery** | Hardcoded in your prompt/code | Standardized: `tools/list` over JSON-RPC | Standardized: agent card at `/.well-known/agent-card.json` |
| **Reusability across hosts** | None — you own all the integration code | High — same server works in Claude, ChatGPT, Cursor, etc. | High — same agent callable by any A2A client |
| **Auth** | Whatever your app already does | OAuth 2.1 for remote, none for stdio | OAuth 2.1 or bearer tokens; signed agent cards for trust |
| **Best for** | One-off, in-process, latency-critical | Tools you'd reuse across multiple AI hosts | Agents you'd compose across vendor boundaries |

### The decision tree

```
Are you calling something that takes minutes-to-hours and has its own
agent loop, plans, intermediate state, can be canceled?
                          │
              ┌───YES─────┴──────NO───┐
              ▼                       ▼
            A2A                Will the same capability be
                               needed by other AI hosts /
                               apps in the future?
                                       │
                          ┌────YES─────┴─────NO────┐
                          ▼                        ▼
                         MCP               Plain function calling
                                           (just write the function)
```

### When function calling is the right answer (and you don't need a protocol)

| Signal | Why function calling, not MCP |
|---|---|
| You're building a single app and the tools are app-specific | No reuse value; protocol overhead isn't worth it |
| Sub-100ms latency budget for tool calls | MCP adds a process boundary + JSON-RPC overhead |
| The "tool" is really just a Python function with no side effects | Wrap it as a LangChain tool, done |
| You're prototyping; will revisit later | Don't pay protocol tax to learn faster |

Don't reach for MCP because "it's the modern way." Reach for it when the cross-host reuse, the standardized discovery, or the secure remote deployment story actually matters to you.

### How they compose in a real 2026 system

The interesting architectures are not "MCP vs A2A" — they're **both, stacked**:

```
                ┌─────────────────────────────────────┐
                │   Orchestrator agent (LangGraph)    │  ← A2A endpoint
                │   In your company, your codebase    │
                └──────────────────┬──────────────────┘
                                   │ A2A delegates to peer agents
       ┌───────────────────────────┼───────────────────────────┐
       │                           │                           │
       ▼                           ▼                           ▼
[Partner's research agent]   [Internal billing agent]   [3rd-party legal agent]
  A2A endpoint                A2A endpoint                A2A endpoint
       │                           │                           │
       │ each agent internally uses tools via MCP             │
       ▼                           ▼                           ▼
  [web search MCP]            [Stripe MCP]               [legal-db MCP]
  [arxiv MCP]                 [internal-CRM MCP]         [contract-db MCP]
  [filesystem MCP]
```

**A2A is the horizontal layer** (agents to agents, often crossing org boundaries).  
**MCP is the vertical layer** (each agent to its tools, often crossing process boundaries).  
**Function calling is the leaf** (each agent's in-process Python tools that didn't justify being MCP servers).

### The classic interview question

> **Q**: *"You're building a customer support agent that needs to (1) look up account info from your CRM, (2) search your help center, (3) delegate complex billing questions to your billing team's agent, (4) escalate legal questions to your law firm's agent. What protocols and where?"*

The clean answer:

- **CRM lookup and help-center search**: MCP servers. They're reusable across hosts (Claude, ChatGPT, your support agent), have stable schemas, deserve session state for streaming long results. Wrap your existing REST APIs as MCP servers.
- **Internal billing agent**: A2A. It's an autonomous agent with its own loop, runs for tens of seconds to minutes, can ask for clarification mid-task.
- **External law firm agent**: A2A with a signed agent card. Same shape as billing, but crossing an org boundary — the signature matters here.
- **In-process utilities** (formatting, slug generation, etc.): plain function calls. No protocol needed.

That's a real-world answer that maps each component to the right layer instead of treating it as either/or.

### A failure mode to know

Teams sometimes try to express agent-to-agent over MCP — wrap the entire other agent as one MCP tool. It works for short interactions, but loses A2A's task lifecycle: no streaming progress, no cancellation, no input-required state for mid-task clarification, no artifact-typed outputs. The other agent looks like a "function that took 30 minutes to return." For anything beyond trivial, you want A2A.


---
# Part B: Coding Agents

## §6 — What makes coding agents structurally different

A coding agent is, at one level, just an agent with tools that read and write files. But the *shape* of the work is so different from chat agents that whole architectural categories exist just for code. Five things make coding agents structurally distinct.

### 1. Long horizons

A chat agent's task usually completes in one turn or a handful of turns. A coding agent's task — "fix this failing test", "add a feature with tests", "refactor this module" — runs for minutes to hours, makes dozens to hundreds of tool calls, and edits many files. The horizon is **10-100× longer**.

This breaks the assumptions of a vanilla ReAct loop. By turn 50, the context is bloated. By turn 100, the agent forgets early decisions. By turn 200, the cost is alarming. So coding agents need:

- **Planning** — explicit task decomposition (write_todos from N1 §8).
- **Context offloading** — write intermediate notes to disk, read them back.
- **Subagents** — isolate long subtasks in their own context windows.

This is why DeepAgents (N1 §16) was extracted from Claude Code: those are the four pillars that make long-horizon coding agents work.

### 2. File-system-as-state

Coding agents don't just read files — they **edit** them. Editing is fundamentally different from reading. Three properties matter:

- **Editing is destructive.** A bad edit deletes work. You need version control as a safety net.
- **Editing is positional.** You can't just say "change the function" — you have to specify *which lines* with surgical precision. This shapes the tool API (see `str_replace`, §8).
- **Editing creates new state to read back.** After an edit, the agent might want to verify by reading the result, then run tests, then iterate. The loop is read → edit → verify → repeat.

### 3. Test loops as the success signal

A chat agent often has no clear success criterion. A coding agent usually does: **the tests pass**. This single property changes everything:

- Reflection becomes powerful — the critic is `pytest`, not another LLM.
- The agent can attempt a fix, observe the test output, refine, and try again. **Self-correction is automatic.**
- Reward signal is sparse but unambiguous — green/red, no judgment call.

Most production coding agents are some variant of: write code → run tests → read failures → fix → repeat. That's why test running (`run_tests`, §8) is in every coding agent's toolkit.

### 4. Sandboxing matters

A coding agent runs *code that an LLM wrote*. Three failure modes follow:

- **Accidentally destructive code** — `rm -rf /tmp` becomes `rm -rf /` if the prompt is off. Without a sandbox, you've lost data.
- **Resource exhaustion** — a fork-bomb-like bug eats your machine.
- **Security escape** — code intended to write to `./data` writes to `~/.ssh/`.

So real coding agents either:
1. **Run in a true sandbox** (container, VM, browser sandbox, cloud isolation) — Devin, Antigravity's terminal sandbox.
2. **Run locally with explicit user approval gates for destructive actions** — Claude Code, Cursor.

There is no third option. "Hope for the best" is not a strategy.

### 5. Diff-first review

A chat agent's output is *the answer.* A coding agent's output is *a diff* — a set of changes against an existing codebase. The diff is the unit of human review.

This means:
- Smaller, focused diffs > sprawling rewrites. A 30-line surgical fix is easier to review than a 500-line "while I was here" cleanup.
- The agent should commit incrementally — git checkpoints are natural HITL gates.
- Diff size is a quality metric in evaluation. "Did the agent fix the bug?" is necessary; "Did it fix the bug without touching unrelated code?" is sufficient.

This is why **Antigravity emphasizes "Artifacts"** (diffs, screenshots, recordings) — the diff itself is the deliverable. The user reviews the artifact, not the agent's chat log.

### The chat agent → coding agent shift in numbers

| Dimension | Chat agent | Coding agent |
|---|---|---|
| Horizon | 1-10 turns | 50-500 turns |
| Tokens per task | 1K-10K | 100K-1M |
| Tools called per task | 0-5 | 20-200 |
| State carried across turns | Conversation history | Filesystem + git state + test results |
| Success signal | Subjective ("looks good") | Objective ("tests pass") |
| Failure mode if you skip planning | Mild rambling | Total flailing across irrelevant files |
| Failure mode if you skip sandboxing | Wasted tokens | Data loss / security incident |
| Right harness | `create_agent` is fine | DeepAgents, or a dedicated coding-agent runtime |

That last row is the key insight. **You don't build a coding agent by stretching a chat-agent harness.** You use a harness designed for long horizons, file edits, and verification loops — that's what every paradigm in §7 is.


## §7 — The four architectures: Devin, Cursor/Antigravity, Claude Code, Augment

Four paradigms dominate coding agents in 2026. Each optimizes a different axis. They're not strictly competing — most serious developers use 2-3 of them for different jobs.

### Paradigm 1: Autonomous cloud (Devin)

**Shape**: A fully sandboxed cloud workspace — its own VM with browser, terminal, editor, filesystem. You give it a task in chat ("fix the OAuth bug in the login flow"); it works for minutes-to-hours autonomously; you check in later.

**Optimization axis**: developer time. The whole point is "fire and forget" — you walk away, do other work, come back to a PR.

**Strengths**:
- True hands-off execution. Run 5 tasks in parallel, none requires your attention.
- The cloud sandbox is genuinely isolated — nothing the agent does affects your machine.
- Suited for tickets that would otherwise sit in your backlog (bug bash, dependency updates, test improvements).

**Weaknesses**:
- **Opaque.** Hard to course-correct mid-task without watching closely.
- **Expensive.** Cloud VMs running for hours add up; Devin's pricing has been a frequent complaint.
- **Sandbox cuts both ways.** Real apps need real credentials, real DBs, real external APIs — sandboxes have to bridge to those carefully.
- **Failure recovery is slow.** When the agent goes off the rails, you've usually paid for the whole run before noticing.

**Best for**: well-scoped backlog tickets, dependency upgrades, prototyping greenfield features overnight, research/exploration that doesn't touch production.

### Paradigm 2: Agent-first IDE (Cursor, Google Antigravity, Windsurf)

**Shape**: An IDE built around the agent, not around code. The editor is one surface; the **agent manager** is the other — a control panel where you spawn, observe, and orchestrate multiple agents working in parallel.

**Optimization axis**: developer throughput, while keeping the developer in the loop. **You're an architect orchestrating agents**, not a typist driving every keystroke.

**Cursor's defining feature** is parallel agents via **git worktrees**. Each agent gets its own worktree (same repo, different working directory, different branch). Five agents can work on five features simultaneously without stepping on each other's files.

**Antigravity's defining feature** (released Nov 2025 alongside Gemini 3) is the **three-surface architecture** plus **Artifacts**:
- Editor surface: traditional code editing with agent sidebar.
- Manager surface: parallel agents working asynchronously, each producing Artifacts.
- Browser surface: the agent has its own Chrome instance for UI testing and validation.

Artifacts are the trust mechanism. Instead of asking you to read the agent's chat log, the agent produces tangible deliverables — task lists, implementation plans, code diffs, screenshots, video recordings of browser testing. You review the artifact, not the trace. **This is the key UX innovation of 2025-2026**: closing the trust gap by making the deliverable visible without making the agent's internal monologue mandatory reading.

Antigravity also has **plan-review-execute** as the default mode (Fast Mode skips planning for trivial tasks), and **Knowledge Items** — patterns the agent learns from feedback and carries across sessions.

**Strengths**:
- Fast iteration — you stay in the loop, agents take the boilerplate.
- Parallel agents multiply throughput without context-switching cost.
- Artifacts make review tractable for long tasks.
- Developer maintains control over critical paths.

**Weaknesses**:
- Tied to the IDE — your terminal scripts and CI can't use this.
- Performance can drag — Antigravity in particular has well-documented latency issues in mid-2026.
- Multi-model parallelism (Antigravity's claim) is impressive on paper but each model still bills separately.

**Best for**: active development, pair-programming style work, frontend iteration where browser preview matters, multi-task workdays where you'd otherwise context-switch a lot.

### Paradigm 3: Terminal / local (Claude Code)

**Shape**: A CLI tool you run in your terminal. No GUI, no cloud sandbox — works directly on your local filesystem via tool calls. Two modes:

- **Plan mode**: the agent reads, thinks, proposes — but doesn't modify anything.
- **Build mode**: the agent executes its plan, with explicit approval gates for destructive actions.

**Optimization axis**: flexibility and scriptability. The agent is *just* a CLI process — you can pipe into it, run it in CI, script it, integrate it with whatever editor or workflow you already have.

**Strengths**:
- Maximum flexibility. Works with any editor, any shell setup.
- Deep filesystem access — the agent can navigate huge repos efficiently.
- **Scriptable** — easy to integrate into CI/CD or batch processing.
- Low overhead per task — no IDE to load.

**Weaknesses**:
- CLI UX. Steeper onboarding for developers who live in VS Code.
- No visual editing or preview — the agent's output is text-only diffs.
- Approval prompts in the terminal are easy to over-approve when tired.

**Best for**: experienced developers, complex multi-file work that doesn't need a browser, automation/scripting, CI integration ("run Claude Code on the failing tests"), terminal-first workflows.

### Paradigm 4: Spec-driven orchestration (Augment Code, Intent)

**Shape**: The central artifact is a **living specification** — a markdown document describing intent, constraints, and acceptance criteria. **Mandatory human approval gates between phases.** Once a phase's spec is approved, parallel specialist agents (planner, implementer, reviewer) execute it.

**Optimization axis**: control and auditability. Spec-driven trades speed for traceability — every code change traces back to an approved spec, and every spec change is reviewed.

**Strengths**:
- **Auditable.** Every change has a spec; every spec was approved. Compliance-friendly.
- **Reduced runaway costs.** The approval gates prevent the agent from spending hours on a misunderstood task.
- Aligns with how enterprise software teams already work (PRDs → tickets → PRs).
- Multiple specialist agents map cleanly to existing team roles.

**Weaknesses**:
- **Heavyweight.** Slow for small tasks where the spec-writing is more work than the change.
- Needs discipline — teams that don't write good specs don't get good code.
- Steepest learning curve of the four.

**Best for**: enterprise software, regulated industries (finance, healthcare), large codebases with many contributors, long-lived features where the spec adds value.


### Paradigm comparison

| | Autonomous Cloud (Devin) | Agent-first IDE (Cursor, Antigravity) | Terminal/Local (Claude Code) | Spec-Driven (Augment) |
|---|---|---|---|---|
| Human involvement | Lowest | High | Medium | Highest at gates, low during execution |
| Setup cost | Low | Medium | Low | High |
| Best task duration | Medium-long | Small-medium | Any | Large |
| Parallelism | Cloud-native | git worktrees / agent manager | Shell parallelism | Parallel specialists |
| Debugging | Hard (opaque) | Easy (artifacts) | Easy (terminal trace) | Easy (spec audit) |
| Cost per task | High | Low-medium | Low | Medium-high |
| Sandbox | Cloud VM | Optional (Antigravity has terminal sandbox) | Local, approval gates | Per-agent, by phase |
| Best at | Fire-and-forget tickets | Active pair-programming | Scripting + CI | Compliance, large teams |

### The decision rule

```
Are you actively coding right now?
   YES → agent-first IDE (Cursor or Antigravity)

Are you scripting / running in CI / want CLI integration?
   YES → terminal (Claude Code)

Do you want to dispatch a task and walk away for an hour?
   YES → autonomous cloud (Devin), or Antigravity Manager view

Do you need an audit trail with approval gates for compliance?
   YES → spec-driven (Augment, Intent)
```

Most working developers use **2-3 of these for different jobs**. Cursor or Antigravity for active development, Claude Code for terminal/scripts/CI, Devin for fire-and-forget tickets, spec-driven for high-stakes changes that need an audit trail.

### The common architecture across all four

Despite the surface differences, every coding agent shares the same core stack. The differences are mostly UX, sandboxing, and orchestration — not fundamental architecture:

```
   ┌────────────────────────────────────────────────────┐
   │                  LLM (the brain)                    │
   │       Claude Sonnet/Opus, Gemini 3 Pro, GPT-5      │
   └────────────────────────┬───────────────────────────┘
                            │ tool calls
                            ▼
   ┌────────────────────────────────────────────────────┐
   │                  Tool layer                         │
   │  view / read_file / write_file / edit / str_replace │
   │  bash / glob / grep / list_dir                      │
   │  run_tests / git ops                                │
   │  (these are MCP servers, in-process tools, or both) │
   └────────────────────────┬───────────────────────────┘
                            │ filesystem operations
                            ▼
   ┌────────────────────────────────────────────────────┐
   │              Sandbox / workspace                    │
   │   Local filesystem | Container | Cloud VM | Worktree│
   └────────────────────────────────────────────────────┘
```

Five components shared across all four paradigms:

1. **Planning phase** — explicit (Augment specs, Antigravity Artifacts) or implicit (Claude Code's plan mode, write_todos).
2. **Tool-calling loop** — filesystem + shell + (often) web as the primary tools.
3. **Context management** — file indexing, retrieval, summarization (§9).
4. **Verification** — tests, linters, type-checkers, in some cases browser-based UI checks.
5. **Rollback capability** — git commits are the natural checkpoints; some sandboxes add VM snapshots.

This is why N1 §16 (DeepAgents) is the right intellectual frame for understanding any coding agent: planning + filesystem + subagents + detailed system prompt. The four paradigms differ in *how* they package those, not in *what* they do.

### Common architecture, different surfaces — the canonical example

The same task — "fix the failing test in `auth/test_login.py`" — looks like this in each paradigm:

| Paradigm | The interaction |
|---|---|
| Devin | Tell it in chat. Walk away. Come back to a PR. |
| Cursor | Open the file. Use Composer with the test failure context. Approve the diff. Run tests in Cursor's terminal. |
| Antigravity | Manager view: dispatch an agent on the failing test. It produces a plan artifact, then a diff artifact, then a screenshot of green tests. Approve the diff. |
| Claude Code | `claude code` in the terminal. Plan mode: it surveys auth files. Build mode: it edits `login.py`. Approve `bash pytest`. Tests pass. Approve commit. |
| Augment | Spec: "Login fails with stale cookies; acceptance: test_login passes, cookie expiry respected." Approve spec. Specialist agents plan + implement + review. Approve final diff. |

Same brain, same tools underneath, different surfaces. Pick the surface that matches your workflow.

### A footnote on Antigravity's release context

Antigravity matters in the 2026 conversation because it's the first **agent-manager-as-primary-interface** IDE from a major vendor. Google released it on **Nov 18, 2025**, the same day as Gemini 3, on a heavily modified VS Code fork (debate exists about whether it's a VS Code fork or a Windsurf fork — Windsurf was acquired by Google). It supports Gemini 3 Pro natively plus Claude Sonnet 4.6 / Opus 4.6 (BYO Anthropic API key) and GPT-OSS-120B. By April 2026 it had ~6% developer adoption — fast for a new IDE.

> **Interview note (Samarth-relevant)**: For an AI Engineer with deep vibe-coding experience, the meta-skill being tested isn't "can you write code" — it's **"can you pick the right agent for the job and run multiple in parallel?"** The framing that lands: "I use Antigravity Manager for parallel work, Claude Code for terminal/CI integration, and Cursor for surgical edits in critical paths. Each optimizes a different axis. Treating them as one category misses the point."


## §8 — The tool kit for coding agents

Every coding agent's tool kit looks roughly the same. The reasons each tool exists, and the *exact shape* of the API, matter — these are the choices that separate a coding agent that flails from one that works.

### The canonical tool list

| Tool | Read or write | Why it exists in this exact shape |
|---|---|---|
| `view` (a.k.a. `read_file`) | Read | View a file's contents with optional line ranges; supports text, images, dirs |
| `glob` | Read | Find files by pattern (`**/*.py`) — fast file discovery |
| `grep` | Read | Search file contents for a regex — fast content discovery |
| `list_dir` (`ls`) | Read | List directory contents; structure discovery |
| `bash` (`shell`, `terminal`) | Write | Run arbitrary shell commands — the escape hatch |
| `str_replace` | Write | Replace a unique substring in a file — surgical, no positional math |
| `create_file` (or `write_file`) | Write | Create a new file or overwrite an existing one |
| `edit` (multi-format) | Write | Higher-level edits; sometimes line-based, sometimes diff-based |
| `run_tests` (or implicit via `bash`) | Read+Write | Execute the test suite, read pass/fail |

Six of these — `view`, `glob`, `grep`, `bash`, `str_replace`, `create_file` — are non-negotiable. Every major coding agent has them. The names vary; the semantics don't.

### Why `str_replace` instead of line-edit?

This is the tool design choice everyone gets wrong on the first try, then gets right. Three options for "edit a file":

1. **Patch / diff input** — agent generates a `diff`, you `git apply` it.
2. **Line-based edit** — agent says "replace lines 42-47 with this."
3. **String-based replace** — agent says "replace this exact substring with this new string."

Why string-based won:

| Approach | Failure mode |
|---|---|
| Patch input | LLM gets line numbers wrong; one off-by-one and the patch fails to apply. Many failures, fragile. |
| Line-based | Same problem — LLM must track line numbers across the whole conversation. Edits at the top of the file shift every line number below. |
| `str_replace` | LLM provides the exact text to find and the text to replace it with. No line numbers; the file does the work. Failure mode: ambiguity (the substring appears more than once) — solved by requiring sufficient context in the search string. |

`str_replace` puts the bookkeeping burden on the file system, not the LLM. **You should not require the LLM to track line numbers across a conversation** — that's what files are for.

### Why two read tools (`glob` + `grep`)?

Because they answer different questions:

- `glob`: "where are the files I care about?" (by filename pattern)
- `grep`: "where is this concept defined?" (by content pattern)

A naive agent without these does `bash find` and `bash cat` — but those are slower, return less-structured output, and don't scale. Dedicated `glob`/`grep` tools return structured results the LLM can reason over directly.

### Why is `bash` necessary at all?

Because no fixed tool set covers everything. The agent will need to run a linter, start a server, run a migration, install a package, query a database. `bash` is the escape hatch.

But `bash` is also the most dangerous tool by far. **Every coding agent's defining safety choice is: how is `bash` constrained?**

| Constraint | Where you'll see it |
|---|---|
| Approval gate before each `bash` call | Claude Code (default) |
| Allowlist of commands that auto-execute | Antigravity's terminal command policy |
| Sandboxed shell (chroot, container) | Devin, Antigravity terminal sandbox |
| Read-only by default; explicit elevation for writes | Some enterprise setups |
| No `bash` at all; only specific tools | Strict spec-driven setups |

The safer the sandbox, the more autonomous the agent can be.

### Tool descriptions matter more than you think

Recall from N1 §3: **the LLM routes between tools by reading descriptions**, not by name. For coding agents specifically:

- Bad: *"Edit a file."*
- Good: *"Replace a unique substring in a file with new content. `old_str` must match the raw file content exactly and appear exactly once. Use this for surgical edits; do not use for whole-file rewrites — use `create_file` for that."*

The second tells the LLM *when not to use the tool* and *what its invariants are*. That's the same "boundary against similar tools" principle from N1 §3, applied at the file-edit layer.

### A toolset for a minimal coding agent

You can build a workable coding agent in a few hundred lines of Python with this set of tools. We'll show them as schemas in the next cell — the schemas are the contract between you and the LLM.


In [4]:
# The schemas of a minimal coding-agent tool kit.
# Same shape used by Claude Code, Cursor, Antigravity (names vary).

from pydantic import BaseModel, Field
from typing import Literal
import json

class ViewArgs(BaseModel):
    """View a file or directory.
    For files: returns the contents (optionally a line range).
    For directories: returns a tree listing up to 2 levels deep."""
    path: str = Field(..., description="Absolute path to the file or directory")
    view_range: tuple[int, int] | None = Field(
        None, description="Optional [start, end] line range; -1 means EOF. Files only."
    )

class GlobArgs(BaseModel):
    """Find files by glob pattern. Fast, returns full paths."""
    pattern: str = Field(..., description="Glob pattern, e.g. '**/*.py' or 'src/auth/*.ts'")
    path: str | None = Field(None, description="Base directory to search; defaults to cwd")

class GrepArgs(BaseModel):
    """Search file contents with a regex. Returns matching lines with file:line context."""
    pattern: str = Field(..., description="Regex pattern")
    path: str | None = Field(None, description="Directory or file to search; defaults to cwd")
    glob: str | None = Field(None, description="Optional file pattern filter, e.g. '*.py'")
    output_mode: Literal["content", "files_with_matches", "count"] = Field("content")

class StrReplaceArgs(BaseModel):
    """Replace a UNIQUE substring in a file.
    old_str must appear EXACTLY ONCE in the file or the call fails.
    Use this for surgical edits. For whole-file rewrites, use create_file."""
    path: str = Field(..., description="Absolute path to the file")
    old_str: str = Field(..., description="Exact substring to replace. Must be unique in the file.")
    new_str: str = Field("", description="Replacement string (empty to delete the old_str)")
    description: str = Field(..., description="Why this edit is being made (for audit)")

class CreateFileArgs(BaseModel):
    """Create a new file or completely overwrite an existing one."""
    path: str = Field(..., description="Absolute path to create")
    file_text: str = Field(..., description="Full file contents")
    description: str = Field(..., description="Why this file is being created")

class BashArgs(BaseModel):
    """Run a shell command. THE ESCAPE HATCH — most dangerous tool.
    Use for: running tests, linters, builds, package installs, git ops.
    Do NOT use for file edits — use str_replace/create_file instead."""
    command: str = Field(..., description="Shell command to run")
    description: str = Field(..., description="Why this command is being run")
    timeout_s: int = Field(60, description="Kill the command after this many seconds")

# Render all of them as the LLM would see them
tools = {
    "view": ViewArgs,
    "glob": GlobArgs,
    "grep": GrepArgs,
    "str_replace": StrReplaceArgs,
    "create_file": CreateFileArgs,
    "bash": BashArgs,
}

print("=== Coding agent tool catalog (as the LLM sees it) ===\n")
for name, schema in tools.items():
    s = schema.model_json_schema()
    desc = (schema.__doc__ or "").strip().split("\n")[0]
    print(f"• {name:14}  {desc}")

print("\n=== Detailed schema for str_replace (the surgical edit tool) ===")
print(json.dumps(StrReplaceArgs.model_json_schema(), indent=2))

print("\nWhy this set of tools, in this shape:")
print("  - str_replace puts the bookkeeping on the FILE, not the LLM.")
print("  - bash is the escape hatch; everything else is purpose-built.")
print("  - Each tool's docstring tells the LLM when NOT to use it.")


=== Coding agent tool catalog (as the LLM sees it) ===

• view            View a file or directory.
• glob            Find files by glob pattern. Fast, returns full paths.
• grep            Search file contents with a regex. Returns matching lines with file:line context.
• str_replace     Replace a UNIQUE substring in a file.
• create_file     Create a new file or completely overwrite an existing one.
• bash            Run a shell command. THE ESCAPE HATCH — most dangerous tool.

=== Detailed schema for str_replace (the surgical edit tool) ===
{
  "description": "Replace a UNIQUE substring in a file.\nold_str must appear EXACTLY ONCE in the file or the call fails.\nUse this for surgical edits. For whole-file rewrites, use create_file.",
  "properties": {
    "path": {
      "description": "Absolute path to the file",
      "title": "Path",
      "type": "string"
    },
    "old_str": {
      "description": "Exact substring to replace. Must be unique in the file.",
      "title": "Old S

## §9 — Context management: filesystem as working memory

Big codebases don't fit in context. A medium repo has 1M-10M tokens of source code. Even Gemini 2.5 Pro's 2M-token context can't hold a real codebase, and you wouldn't want to pay to put it all in even if you could. **Coding agents need a context strategy.**

Three strategies, often combined:

### 1. Agentic exploration (Claude Code's default)

The agent navigates the codebase the same way a developer would: `list_dir` → `glob` for relevant files → `view` the interesting ones → `grep` for specific symbols. The agent reads what it needs, when it needs it.

**Why it works**: the LLM is already good at "where would I expect this code to live?" — naming conventions, file layouts, framework patterns. Trust the model to navigate.

**When it fails**: very large codebases with non-obvious structure, monorepos with deep nesting, repos with poor naming. The agent burns turns just trying to find things.

### 2. Retrieval (Cursor, Windsurf, Antigravity)

The IDE indexes the codebase upfront — embeds every file (or function, or symbol) and stores the vectors. At query time, retrieve the top-K most relevant chunks for the agent's current task.

**Why it works**: skips the "where do I look" phase entirely. The agent starts with the relevant code already in context.

**When it fails**: the embedding misses what the agent actually needs (it returned `auth/login.py` but the bug is in `middleware/cors.py`). Stale indexes after refactors. Embeddings don't capture call-graph relationships.

### 3. Symbol indexing (advanced)

Tree-sitter-based parsers build a call graph and symbol table. The agent queries semantically: "where is `authenticateUser` defined? called? imported?" Languages with strong type systems benefit most.

**Why it works**: gives the agent a precise *structural* view of the code, not just textual proximity.

**When it fails**: dynamically typed code, metaprogramming, configuration-heavy frameworks where the call graph is partly runtime-determined.

### The combined approach in production

The best coding agents combine all three:

```
Task arrives
   │
   ▼
[Retrieval]  → seed with top-K relevant files based on the task description
   │
   ▼
[Symbol index] → if needed, resolve "where is X defined" semantically
   │
   ▼
[Agentic exploration] → for everything else, agent does glob/grep/view itself
```

### Filesystem-as-working-memory (DeepAgents / Claude Code pattern)

A second use of the filesystem, beyond just *reading* the code: the agent **writes intermediate notes**. From DeepAgents (N1 §16) and Claude Code, the pattern is:

- Write the **plan** to `plan.md`.
- Write **partial findings** as you go: `notes/auth_audit.md`, `notes/test_failures.md`.
- Read them back later instead of recomputing.

The filesystem becomes the agent's **extended brain** — anything that would have blown up the context window now lives in a file. This is especially powerful for subagents: a subagent's findings get written to a file; the lead agent reads the file; the subagent's full context never has to merge with the lead's.

### Summarization middleware

When the conversation itself grows too big (lots of tool calls, lots of observations), summarize the older portion. LangChain has `SummarizationMiddleware`; Anthropic's API has prompt caching that helps too. The pattern is the same as §11 in N1 — summary-buffer applied to coding-agent loops.

### Context bloat: the dominant failure mode

When a coding agent's context is mismanaged, this is what you see:

1. The agent spends the first 30 turns exploring the repo (good).
2. By turn 80, the context is bloated with tool calls + observations, much of it stale.
3. The agent re-reads files it already read, because it doesn't remember.
4. By turn 150, the agent contradicts decisions it made at turn 40.
5. By turn 200, the agent is flailing — cost goes up, progress goes down.

The fix is the four pillars from N1 §16: **explicit plan in a file**, **filesystem for offloading**, **subagents for isolation**, and **summarization of old turns**.

> **Interview note.** *"How do you keep a coding agent from losing the plot on a long task?"* Three answers, in order of impact: (1) write the plan to a file (`plan.md`), have the agent re-read it every N turns. (2) Use subagents for any subtask that involves more than 10 tool calls — keeps the lead's context clean. (3) Summarize older turns once total tokens cross 50% of context. Without these, the agent dies in a loop.


## §10 — Failure modes of coding agents

A small set of failure modes accounts for most of the bad coding-agent outcomes. Knowing them by name lets you diagnose fast and prevent them upfront.

### 1. Context overflow → flailing

**Symptom**: agent does great for the first 50 turns, then progressively loses the thread. Re-reads files, contradicts earlier decisions, edits the wrong files.

**Root cause**: context bloat with no summarization. The relevant context gets pushed out by stale tool calls and observations.

**Fix**: from §9 — plan in a file, summarize older turns, use subagents for isolated subtasks.

### 2. Wrong-file edit

**Symptom**: agent edits `tests/test_auth.py` when the bug is in `auth/login.py`. Tests *still* fail (or pass for the wrong reason).

**Root cause**: tool selection error at the file-discovery step. The agent's `glob`/`grep` returned a plausible-but-wrong file, and the agent didn't verify.

**Fixes**:
- Tools should prefer **dual-source confirmation** — find a symbol via grep AND verify by viewing the file before editing.
- Test against an explicit acceptance criterion ("test_login should pass"), not against a vague signal ("the bug should be fixed").
- Stricter system prompt: "before editing, view the file in full and explain why this is the right edit."

### 3. Test-loop fixation

**Symptom**: agent makes a tiny edit, runs tests, sees a different failure, makes another tiny edit, runs tests again, sees another different failure. Five iterations in, the test is *still* red but the agent has touched 12 files.

**Root cause**: the agent treats every test failure as the canonical truth and edits whatever it thinks fixes *that specific failure* — without stepping back to verify the underlying logic is right. It's reactive, not analytical.

**Fixes**:
- Force the agent to write down the **diagnosis** before the fix ("the bug is X because Y; the fix is Z because W").
- Cap the number of consecutive `bash pytest` calls before requiring planning. If the loop spins, force `write_todos` to re-plan.
- Reflection layer: after every failed test run, the agent must explain *why* this approach is different from the last.

### 4. Unsafe destructive actions

**Symptom**: agent runs `rm -rf node_modules` and then can't recover. Or `git reset --hard` losing uncommitted work. Or `DROP TABLE users` against the wrong DB.

**Root cause**: `bash` is a fire-hose with no guard. The agent had access to a destructive command and called it confidently.

**Fixes**:
- **Approval gates on `bash`** — terminal coding agents (Claude Code, Cursor) require explicit approval for shell commands by default. Don't disable this.
- **Sandbox** the workspace — Devin and Antigravity's terminal sandbox prevent destructive escape.
- **Snapshot before changes** — git commits, file backups, container snapshots. Easy rollback.
- **Allowlist destructive commands** — `rm -rf`, `DROP`, `git reset --hard`, `chmod 777`, `sudo`. Require explicit user approval even if everything else auto-runs.

### 5. Hallucinated APIs / packages

**Symptom**: agent confidently writes `import some_package as sp` and then `sp.do_the_thing(args)` — except `some_package` doesn't exist, or doesn't have that function.

**Root cause**: LLM training data is finite; new APIs (or APIs renamed in recent versions) get hallucinated.

**Fixes**:
- **Verify before using.** Tool: "look up this package on PyPI before importing." Or: run `pip list` to confirm what's installed.
- **Test-driven**: if the test fails with ImportError, the agent catches the hallucination immediately and corrects.
- **Pin the documentation** — give the agent access to up-to-date docs via MCP server (Context7, official package docs).

### 6. Diff explosion ("while I was here" cleanup)

**Symptom**: you asked for a 5-line bug fix; the diff is 300 lines because the agent also "improved" unrelated code.

**Root cause**: no constraint on diff scope. The agent's reward signal (the test passes) doesn't penalize over-editing.

**Fixes**:
- Explicit instruction in the system prompt: "make the smallest change that fixes the bug. Do not touch code outside the failing path unless directly required."
- **Diff size** as an evaluation metric (covered in N3 §3).
- **Approval review** — every diff over N lines requires human approval. Antigravity's Artifact review system surfaces this naturally.

### 7. Silent failure / fake success

**Symptom**: agent reports "done!" but the change didn't actually fix the bug, or the tests it "passed" weren't the right tests.

**Root cause**: the agent's verification step is sloppy. It ran *some* tests, saw they passed, declared victory — without confirming the *right* tests ran.

**Fixes**:
- **Explicit acceptance criteria** in the task description: "test_login_with_expired_cookie must pass."
- **Run the entire test suite**, not just the changed module's tests.
- **Final verification step** in the agent's plan: "confirm the original failing test now passes and no other tests broke."

### A failure-mode triage table

| Symptom | Most likely cause | First thing to check |
|---|---|---|
| Agent flails after 50+ turns | Context overflow | Is there a plan file? Is summarization on? |
| Agent edited wrong file | Bad discovery / tool selection | Did agent view the file before editing? |
| Stuck in test loop | Reactive editing without diagnosis | Force write_todos between test runs |
| Lost uncommitted work | Unsafe destructive action | Are bash approval gates on? |
| ImportError or AttributeError | Hallucinated API | Is doc-MCP server connected? Verify with `pip list`? |
| 500-line diff for a 5-line bug | No scope constraint | Add scope rules to system prompt; review artifact |
| "Done" but bug persists | Sloppy verification | Was the original failing test re-run? |

> **Interview note.** *"What's the failure mode that worries you most about giving an agent shell access?"* The answer is destructive actions you can't undo — `rm -rf`, `git reset --hard`, unrecoverable DB ops. The mitigation stack is: approval gates → sandbox → snapshots → allowlist destructive commands. State all four, in that order; it's the right defense-in-depth answer.


---
# Synthesis & Cheat Sheet

## The protocol stack you should be able to draw on a whiteboard

```
                  [User]
                    │
                    ▼
              [AI Host App]                    e.g. Claude, ChatGPT, Cursor,
              ┌─────┴─────┐                          Claude Code, Antigravity
              │   ↑       │
              │   │  in-process tools         function calling
              │   ▼
              │  [Agent loop]
              │   │   │
              │   │   └────────────────── A2A ──► [Peer agent]   long-running task
              │   │                                              streaming artifacts
              │   │                                              cancellable
              │   │
              │   └────────── MCP ──────► [MCP server 1]    tools / resources / prompts
              │                            [MCP server 2]    stdio or Streamable HTTP
              │                            [MCP server N]    OAuth 2.1 for remote
              │
              └─→ MCP servers may be local processes or remote HTTPS endpoints
```

## Cheat sheet

### Quick MCP facts

| | |
|---|---|
| Origin | Anthropic, November 2024 |
| Governance | Linux Foundation / Agentic AI Foundation, donated Dec 2025 |
| Transport | stdio (local), Streamable HTTP (remote). HTTP+SSE deprecated March 2025 |
| Auth (remote) | OAuth 2.1 with PKCE + Resource Indicators (RFC 8707) |
| Primitives | Tools (verbs) / Resources (nouns) / Prompts (recipes) |
| Discovery | `tools/list`, `resources/list`, `prompts/list` over JSON-RPC |
| Wire protocol | JSON-RPC 2.0 |
| Roles | Host runs N clients; each client connects to 1 server |
| SDK downloads | ~97M/month (early 2026) |
| Roadmap 2026 | Stateless sessions for horizontal scale, enterprise auth, MCP Server Cards |

### Quick A2A facts

| | |
|---|---|
| Origin | Google, April 2025 |
| Governance | Linux Foundation, donated June 2025 |
| v1.0 changes | Signed agent cards, gRPC bindings, multi-tenancy, version negotiation |
| Discovery | Agent Card at `/.well-known/agent-card.json` |
| Verb | `tasks/send` — long-running, stateful tasks (not request/response) |
| Task states | submitted → working → completed / failed / canceled / input-required |
| Adoption | 150+ orgs in production (April 2026) |
| Key idea | "MCP for tools, A2A for agents" |

### Decision: which protocol when

| Calling | Use |
|---|---|
| In-process Python function | Function calling |
| Tool that other AI hosts will also need | MCP |
| Long-running agent with its own loop | A2A |
| Inside-the-same-codebase multi-agent | LangGraph supervisor/swarm (not a protocol) |

### Coding agent paradigms in a sentence each

- **Devin** — autonomous cloud VM; fire-and-forget tickets
- **Cursor** — agent-first IDE with parallel agents via git worktrees
- **Antigravity** — agent-first IDE with Manager surface, Artifacts, Browser sandbox, Knowledge Items
- **Claude Code** — terminal-first; plan mode + build mode + approval gates
- **Augment / Intent** — spec-driven; mandatory approval gates between phases

### The coding-agent tool kit (every paradigm has these)

`view`, `glob`, `grep`, `list_dir`, `bash`, `str_replace`, `create_file`, `run_tests`

### Numbers and facts worth remembering

| Number | What |
|---|---|
| Nov 2024 | MCP introduced |
| Dec 2025 | MCP donated to Linux Foundation |
| Nov 18, 2025 | Antigravity released alongside Gemini 3 |
| Apr 2025 | A2A announced by Google |
| Jun 2025 | A2A donated to Linux Foundation |
| Early 2026 | A2A v1.0 — signed agent cards |
| 2 | Number of MCP transports (stdio + Streamable HTTP) |
| 3 | MCP primitives (tools, resources, prompts) |
| 4 | Coding agent paradigms |
| ~97M | MCP SDK monthly downloads (early 2026) |
| 150+ | A2A organizations in production |
| OAuth 2.1 | Standard for both MCP remote auth and A2A auth |

## Interview narration

### 90-second MCP answer

> "MCP is an open protocol for connecting LLMs to external tools and data, introduced by Anthropic in November 2024 and donated to the Linux Foundation in December 2025. It's JSON-RPC 2.0 over stdio or Streamable HTTP, with three primitives: tools (verbs the LLM calls), resources (read-only data the LLM attaches to context), and prompts (reusable templates the user invokes). The architecture is host + clients + servers — the host runs N clients, one per server, each connected to a separate process. The killer property is reuse: one MCP server (say, `github-mcp`) works in Claude, ChatGPT, Cursor, VS Code Copilot — anywhere that speaks MCP. The protocol replaces the M × N integration matrix with M + N. For remote servers, auth is OAuth 2.1 with PKCE and Resource Indicators. The 2026 roadmap focuses on stateless sessions for horizontal scaling and `.well-known` discovery metadata."

### 90-second A2A answer

> "A2A is the agent-to-agent protocol — Google's answer for how autonomous agents from different vendors and frameworks discover and coordinate. Introduced April 2025, donated to the Linux Foundation in June 2025, v1.0 in early 2026. The discovery primitive is the agent card at `/.well-known/agent-card.json` — describes the agent's name, capabilities, skills, auth requirements. v1.0 added signed cards so an agent can verify the card came from the claimed domain. The protocol's main verb is `tasks/send` — long-running tasks with a lifecycle (submitted → working → completed / failed / canceled / input-required). That's the key difference from MCP: MCP is request/response for tool calls; A2A is task-stateful for agent delegation. The official guidance: MCP for tools, A2A for agents. They compose — your top-level A2A coordinator delegates to peer agents, each of those uses MCP internally for tools."

### 90-second coding-agent answer

> "Four paradigms in 2026, each optimizing a different axis. **Autonomous cloud** like Devin — cloud VM, fire-and-forget, you walk away for an hour. **Agent-first IDE** like Cursor and Google Antigravity — Manager surface dispatches multiple agents in parallel via git worktrees, with Artifacts as the trust mechanism. **Terminal/local** like Claude Code — CLI, plan mode + build mode + explicit approval gates for destructive actions, scriptable and CI-friendly. **Spec-driven** like Augment or Intent — living specification + approval gates between phases, audit-friendly for enterprise. Under the hood, all four share the same architecture: an LLM, a tool layer with view/glob/grep/bash/str_replace/create_file/run_tests, and a sandbox of some shape. The differences are surface and sandbox, not architecture. The interesting failure modes are context overflow, wrong-file edits, test-loop fixation, and unsafe destructive actions — each has a clean mitigation in the §10 table."

### Connecting to your past work (Samarth-specific)

For your Seller Copilot work, the resonance is **tool design at scale**. You used tool retrieval to keep the agent's tool set bounded — the same problem MCP solves at the protocol layer (don't show the LLM 100 tools; let it discover what's relevant). The bridge:

> "On Seller Copilot, we hit the tool-selection-at-scale problem early — too many tools in context degraded the LLM's accuracy. We used embedding-based tool retrieval. If I were building it today, I'd push that one layer up: wrap each tool category as an MCP server, let the host's tool retrieval pick a server first, then that server's `tools/list` is the bounded set. Same idea, different layer."

---

*End of Notebook 2.*

**Next: N3 — Evaluation (trajectory eval, tool-call eval, parallel analysis, LLM-as-judge for agents), Serving (sync/async/streaming/background, state stores, durable execution), Monitoring (tracing, alerting), Guardrails (input/output/resource), and the maintain-an-agent loop.**
